In [1]:
##imports
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [35]:
##Load Dataset
import pandas as pd
DATASET_PATH='dataset/'
csv_path=DATASET_PATH+"test_dataset.csv"

df=pd.read_csv(csv_path)


df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 150 non-null    int64 
 1   Title              150 non-null    object
 2   Poet               150 non-null    object
 3   text               150 non-null    object
 4   ctext              150 non-null    object
 5   Poem Link          150 non-null    object
 6   our_summary        150 non-null    object
 7   prompt_by_summary  150 non-null    object
 8   video_path         12 non-null     object
 9   prompt_by_ctext    150 non-null    object
dtypes: int64(1), object(9)
memory usage: 11.8+ KB


,id,Title,Poet,text,ctext,Poem Link,our_summary,prompt_by_summary,video_path,prompt_by_ctext
0,0,"Dear John, Dear Coltrane by Michael S. Harper",Michael S. Harper,"'Dear John, Dear Coltrane' by Michael S. Harpe...","a love supreme, a love supreme\na love supreme...",https://www.poetryfoundation.org/poems/42827/d...,"The poem explores themes of love, loss, pain, ...",A lone musician stands on a dimly lit stage un...,videos/0.mp4,"A person stands in a bustling marketplace, sur..."
1,1,Parrot by Stevie Smith,Stevie Smith,‘Parrot‘ depicts the declining health of a won...,The old sick green parrot\nHigh in a dingy cag...,https://revise.wales/pastPapers/A-level/Englis...,"This old parrot, sick and full of rage, longs ...",A weathered and aged parrot with vibrant yet m...,videos/1.mp4,"Scene 1: A sick green parrot with dull, dishev..."
2,2,Dust of Snow by Robert Frost,Robert Frost,"The simplicity, in the end, is the key element...",The way a crow\nShook down on me\nThe dust of ...,https://www.poetryfoundation.org/poems/44262/d...,The sight of a crow shaking snow from a tree t...,"A solitary poet, bundled in a woolen coat and ...",videos/2.mp4,"A person stands in a quiet, natural setting su..."
3,3,Suburban Sonnet by Gwen Harwood,Gwen Harwood,'Suburban Sonnet' by Gwen Harwood is a poem ab...,"She practises a fugue, though it can matter\nt...",https://genius.com/Gwen-harwood-suburban-sonne...,"A mother practices music, but her children int...",A mother sits at a grand piano in a cozy livin...,videos/3.mp4,Scene 1:\nA woman practices playing a keyboard...
4,4,Unending Love by Rabindranath Tagore,Rabindranath Tagore,'Unending Love' by Rabindranath Tagore is a he...,"I seem to have loved you in numberless forms, ...",https://allpoetry.com/Unending-Love,The speaker expresses their eternal love for s...,A person stands in a serene open field under a...,videos/4.mp4,A serene stream flows gently under a starry ni...


In [3]:
model_path = '../models/VPO-5B'  # replace with your model path
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

model = AutoModelForCausalLM.from_pretrained(model_path).half().eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [30]:
# ------------------------------
# 1. Prompt Template
# ------------------------------
prompt_template = """
Your task is to convert the given poem into a strictly visual, scene-by-scene video prompt.
Focus ONLY on what the poem shows visually. Avoid interpretation, added emotions, symbolism, or story expansion.

------------------------------
RULES
------------------------------

1. **Visual-Only Representation**
   - Describe ONLY elements directly visible in the poem: people, objects, actions, or locations clearly mentioned.
   - Do NOT add new props, décor, rooms, scenery, lighting, emotions, or symbolic meaning.
   - No invented details (e.g., “cluttered room,” “she looks tired”) unless explicitly in the poem.

2. **Scene Creation (1–5 Scenes)**
   - Create a new scene ONLY when the poem clearly shifts visually.
   - Keep scenes minimal if the poem is visually simple.
   - Each scene MUST include:
       • setting (ONLY if poem gives it)  
       • characters mentioned  
       • visible actions only  
       • visible objects only  
       • natural atmosphere (e.g., “steam rises”) — NOT emotional tone  
   - NO cinematic commands (“camera shifts…”) unless visually implied.

3. **No Emotional Interpretation**
   - Do NOT describe feelings unless the poem visually shows them through explicit actions.
   - If emotions are implied but not visible, ignore them.

4. **No Story Expansion**
   - Do NOT add backstory, memories, motivations, or extra narrative steps.
   - Do NOT add furniture, rooms, clothing, or scenery unless they appear clearly in the poem.

5. **Optional Indian Context**
   - ONLY adapt visuals into an Indian setting if it naturally fits the imagery.
   - NEVER force Indian elements.

6. **High-Quality Video Readability**
   - Use clear, straightforward visual descriptions.
   - Avoid metaphors unless they are literal visuals.
   - Make scenes easy for a video model to generate.

------------------------------
EXAMPLE
------------------------------

Example Poem:
"The way a crow  
Shook down on me  
The dust of snow  
From a hemlock tree"

Example Output (Scene-by-Scene):
Scene 1:
A crow sits on the branch of a hemlock tree covered in light snow.

Scene 2:
The crow moves suddenly, shaking a small amount of snow off the branch.

Scene 3:
A person stands below the tree as the dust of snow falls onto their head and shoulders.

------------------------------
INPUT:
Poem:
{}

OUTPUT:
Video Prompt (Scene-by-Scene):
"""




# ------------------------------
# 2. Function to Generate Video Prompt
# ------------------------------
def generate_video_prompt(text):
    message = [{'role': 'user', 'content': prompt_template.format(text)}]

    # ✅ Build prompt string
    prompt = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=False
    )

    # ✅ Tokenize into a dictionary (correct input format)
    model_inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    # ✅ Generate
    with torch.no_grad():
        output = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            do_sample=True,
            top_p=1.0,
            temperature=0.7,
            num_beams=1
        )

    # ✅ Decode clean text
    decoded = tokenizer.decode(output[0])

    # ✅ Extract only assistant response
    if "<|start_header_id|>assistant<|end_header_id|>" in decoded:
        decoded = decoded.split("<|start_header_id|>assistant<|end_header_id|>", 1)[1]

    if "<|eot_id|>" in decoded:
        decoded = decoded.split("<|eot_id|>", 1)[0]

    return decoded.strip()



In [32]:
print(df.iloc[3]['Poem Link'])
print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

print(df.iloc[3]['ctext'])
print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
# print(df.iloc[3]['prompt_by_summary'])
# print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
print(generate_video_prompt(df.iloc[3]['ctext']))

# generate_video_prompt(df.iloc[0]['our_summary'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


https://genius.com/Gwen-harwood-suburban-sonnet-annotated
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
She practises a fugue, though it can matter
to no one now if she plays well or not.
Beside her on the floor two children chatter,
then scream and fight. She hushes them. A pot
boils over. As she rushes to the stove
too late, a wave of nausea overpowers
subject and counter-subject. Zest and love
drain out with soapy water as she scours
the crusted milk. Her veins ache. Once she played
for Rubinstein, who yawned. The children caper
round a sprung mousetrap where a mouse lies dead.
When the soft corpse won't move they seem afraid.
She comforts them; and wraps it in a paper
featuring: Tasty dishes from stale bread.
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Scene 1:
A woman sits at a piano, her fingers moving mechanically as she practices a fugue. The room appears modest, with a wooden floor and simple furniture. Two children sit nearby, occasionall

In [33]:
# out_col="prompt_by_summary"
# in_col="our_summary"
in_col="ctext"
out_col="prompt_by_ctext"
def process_dataset_in_batches(df, output_file=csv_path, batch_size=20, text_column=in_col):
    # 1. Ensure output column exists
    if out_col not in df.columns:
        df[out_col] = ""

    total_rows = len(df)

    for start_idx in range(0, total_rows, batch_size):
        end_idx = min(start_idx + batch_size, total_rows)

        # Extract the batch (copy so we don't modify df by mistake)
        batch = df.iloc[start_idx:end_idx].copy()

        # 2. Skip rows that already have text in prompt_by_summary
        mask_needed = batch[out_col].astype(str).str.strip() == ""
        batch_to_process = batch[mask_needed]

        if batch_to_process.empty:
            print(f"Skipping rows {start_idx}–{end_idx-1}: already processed")
            continue

        # 3. Process only needed rows
        batch.loc[mask_needed,out_col] = (
            batch_to_process[text_column].apply(generate_video_prompt)
        )

        # 4. Update original df for only processed rows
        df.loc[start_idx:end_idx-1, out_col] = batch[out_col]

        print(f"Processed rows {batch_to_process.index.tolist()}")

    # 5. Save entire df only once at the end
    df.to_csv(output_file, index=False)
    print("All batches processed and saved!")

In [34]:
process_dataset_in_batches(df,batch_size=10)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed rows [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed rows [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed rows [120, 121, 122, 123, 124, 125, 126, 127, 128, 129]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed rows [130, 131, 132, 133, 134, 135, 136, 137, 138, 139]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed rows [140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
All batches processed and saved!
